In [1]:
import pandas as pd
import numpy as np

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
df = pd.read_csv("SampleSuperstore.csv")

print("Dataset loaded successfully")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully
Rows: 9994
Columns: 13


In [3]:
print("FIRST 5 ROWS")
display(df.head())

print("\nDATASET INFORMATION")
df.info()

print("\nMISSING VALUES")
print(df.isnull().sum())

print("\nDUPLICATE ROWS")
print(df.duplicated().sum())

FIRST 5 ROWS


,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.9600,2,0.00,41.9136
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.9400,3,0.00,219.5820
2,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.6200,2,0.00,6.8714
3,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.5775,5,0.45,-383.0310
4,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.3680,2,0.20,2.5164



DATASET INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Ship Mode     9994 non-null   object 
 1   Segment       9994 non-null   object 
 2   Country       9994 non-null   object 
 3   City          9994 non-null   object 
 4   State         9994 non-null   object 
 5   Postal Code   9994 non-null   int64  
 6   Region        9994 non-null   object 
 7   Category      9994 non-null   object 
 8   Sub-Category  9994 non-null   object 
 9   Sales         9994 non-null   float64
 10  Quantity      9994 non-null   int64  
 11  Discount      9994 non-null   float64
 12  Profit        9994 non-null   float64
dtypes: float64(3), int64(2), object(8)
memory usage: 1015.1+ KB

MISSING VALUES
Ship Mode       0
Segment         0
Country         0
City            0
State           0
Postal Code     0
Region          0
Category        0
Su

In [4]:
duplicates = df[df.duplicated(keep=False)]

print("Number of duplicate rows:", len(duplicates))
display(duplicates)


Number of duplicate rows: 34


,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
568,Standard Class,Corporate,United States,Seattle,Washington,98105,West,Office Supplies,Paper,19.440,3,0.0,9.3312
591,Standard Class,Consumer,United States,Salem,Oregon,97301,West,Office Supplies,Paper,10.368,2,0.2,3.6288
935,Standard Class,Home Office,United States,Philadelphia,Pennsylvania,19120,East,Office Supplies,Paper,15.552,3,0.2,5.4432
950,Standard Class,Home Office,United States,Philadelphia,Pennsylvania,19120,East,Office Supplies,Paper,15.552,3,0.2,5.4432
1186,Standard Class,Corporate,United States,Seattle,Washington,98103,West,Office Supplies,Paper,25.920,4,0.0,12.4416
1479,Standard Class,Consumer,United States,San Francisco,California,94122,West,Office Supplies,Paper,25.920,4,0.0,12.4416
2803,Standard Class,Consumer,United States,San Francisco,California,94122,West,Office Supplies,Paper,12.840,3,0.0,5.7780
2807,Second Class,Consumer,United States,Seattle,Washington,98115,West,Office Supplies,Paper,12.960,2,0.0,6.2208
2836,Standard Class,Consumer,United States,Los Angeles,California,90036,West,Office Supplies,Paper,19.440,3,0.0,9.3312
3127,Standard Class,Consumer,United States,New York City,New York,10011,East,Office Supplies,Paper,49.120,4,0.0,23.0864


In [5]:
print("Duplicate row count before cleaning:", df.duplicated().sum())

Duplicate row count before cleaning: 17


In [6]:
# Store the original number of rows
rows_before = len(df)

# Remove exact duplicate rows
df = df.drop_duplicates()

# Store the new number of rows
rows_after = len(df)

print("Rows before cleaning:", rows_before)
print("Rows after removing duplicates:", rows_after)
print("Duplicate rows removed:", rows_before - rows_after)

Rows before cleaning: 9994
Rows after removing duplicates: 9977
Duplicate rows removed: 17


In [7]:
# Check for leading/trailing spaces in text columns

text_columns = df.select_dtypes(include="object").columns

for col in text_columns:
    spaces = (df[col] != df[col].str.strip()).sum()
    print(f"{col}: {spaces} values with extra spaces")

Ship Mode: 0 values with extra spaces
Segment: 0 values with extra spaces
Country: 0 values with extra spaces
City: 0 values with extra spaces
State: 0 values with extra spaces
Region: 0 values with extra spaces
Category: 0 values with extra spaces
Sub-Category: 0 values with extra spaces


In [8]:
# Check numerical columns for invalid values

print("Negative Sales:", (df["Sales"] < 0).sum())
print("Invalid Quantity (<= 0):", (df["Quantity"] <= 0).sum())
print("Invalid Discount (< 0 or > 1):",
      ((df["Discount"] < 0) | (df["Discount"] > 1)).sum())

print("\nMinimum values:")
print(df[["Sales", "Quantity", "Discount", "Profit"]].min())

print("\nMaximum values:")
print(df[["Sales", "Quantity", "Discount", "Profit"]].max())

Negative Sales: 0
Invalid Quantity (<= 0): 0
Invalid Discount (< 0 or > 1): 0

Minimum values:
Sales          0.444
Quantity       1.000
Discount       0.000
Profit     -6599.978
dtype: float64

Maximum values:
Sales       22638.480
Quantity       14.000
Discount        0.800
Profit       8399.976
dtype: float64


In [9]:
# Detect outliers using the IQR method

numeric_columns = ["Sales", "Quantity", "Discount", "Profit"]

for col in numeric_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    print(f"{col}")
    print(f"  Lower bound: {lower_bound:.2f}")
    print(f"  Upper bound: {upper_bound:.2f}")
    print(f"  Outliers: {len(outliers)}")
    print()

Sales
  Lower bound: -271.70
  Upper bound: 498.98
  Outliers: 1167

Quantity
  Lower bound: -2.50
  Upper bound: 9.50
  Outliers: 170

Discount
  Lower bound: -0.30
  Upper bound: 0.50
  Outliers: 855

Profit
  Lower bound: -39.74
  Upper bound: 70.84
  Outliers: 1881



In [10]:
# Final validation after cleaning

print("FINAL DATA QUALITY CHECK")
print("=" * 40)

print("\nDataset shape:")
print(df.shape)

print("\nMissing values:")
print(df.isnull().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

print("\nNumeric validation:")
print("Negative Sales:", (df["Sales"] < 0).sum())
print("Invalid Quantity:", (df["Quantity"] <= 0).sum())
print("Invalid Discount:",
      ((df["Discount"] < 0) | (df["Discount"] > 1)).sum())

FINAL DATA QUALITY CHECK

Dataset shape:
(9977, 13)

Missing values:
0

Duplicate rows:
0

Data types:
Ship Mode        object
Segment          object
Country          object
City             object
State            object
Postal Code       int64
Region           object
Category         object
Sub-Category     object
Sales           float64
Quantity          int64
Discount        float64
Profit          float64
dtype: object

Numeric validation:
Negative Sales: 0
Invalid Quantity: 0
Invalid Discount: 0


In [11]:
# Save the cleaned dataset

output_file = "cleaned_superstore.csv"

df.to_csv(output_file, index=False)

print("Cleaned dataset saved successfully as:", output_file)

Cleaned dataset saved successfully as: cleaned_superstore.csv


In [12]:
cleaned_check = pd.read_csv("cleaned_superstore.csv")

print("Rows:", cleaned_check.shape[0])
print("Columns:", cleaned_check.shape[1])
print("Duplicate rows:", cleaned_check.duplicated().sum())
print("Missing values:", cleaned_check.isnull().sum().sum())

Rows: 9977
Columns: 13
Duplicate rows: 0
Missing values: 0
